# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

This stage tracks two distinct wallet groups on the full trade stream (BUY + SELL):

- `copyable_group`: wallets that pass base eligibility and strong copyability thresholds.
- `predicting_group`: the remaining eligible wallets (not in copyable) that pass a predictability score threshold.

Notes:
- Group membership is threshold-based (absolute scores), not target-count based.
- Open-buy skill metrics are kept for diagnostics/legacy comparison.
- `selected_wallets_v2.parquet` stores both groups with a `wallet_group` label.
- Stage2 backtest uses only `predicting_group`.


In [4]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [5]:
from enum import Enum

class MarketFilterMode(Enum):
    DEFAULT = "default"
    SPORTS = "sports"
    CRYPTO = "crypto"

# ── MARKET FILTER MODE SWITCH ────────────────────────────────────────────────
MARKET_FILTER_MODE = MarketFilterMode.SPORTS

# ── configuration ─────────────────────────────────────────────────────────────
# TAGS = set(['Weather'])
# TAGS = set(['Esports'])
TAGS = set(['Sports'])
# TAGS = set(['Awards'])
# TAGS = set(['MrBeast', 'Trump', 'Elections', 'Celebrities', 'Elon Musk', 'YouTube'])

RECENCY_DAYS = 90

MAX_LEAD_DAYS = 2
if 'Politics' in TAGS:
    MAX_LEAD_DAYS = 14

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    TRADES_DIR    = Path("../../data/polygon_trades_processed_sports")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_sports")
elif MARKET_FILTER_MODE == MarketFilterMode.CRYPTO:
    TRADES_DIR    = Path("../../data/polygon_trades_processed_crypto")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_crypto")
else:
    TRADES_DIR    = Path("../../data/polygon_trades_processed")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")


WALLET_FILTER = None
# WALLET_FILTER = [
#     '0x919698b19427cbe6945b0dc823f2d9e126a4d934',
#     '0x85d22a973e817f87afe0a7fd346a39823cce7479',
#     '0x6011655c4afb76f36dd1b08a137a1ba73466b31e',
#     '0x5f612351a9e46afbe3f164473a7d77a69ec9840a',
# ]
# ── exposure thresholds (bankroll-discipline carve, signal_lab §14-18) ─────
# Copy only trades whose per-wallet exposure (position * price) is >= T_{w,f},
# the largest exposure capturing fraction `f` of the wallet's train copyable PnL
# (fit via wallet_capture_thresholds on train; f == 1.0 copies everything).
EXPOSURE_FRACS   = (1, 0.5, 0.2) #, 0.95, 0.9, 0.8, 0.5, 0.2, 0.1, 0.05)
EXPOSURE_PNL_COL = "copyable_pnl"
EXPOSURE_EXP_COL = "exposure"   # computed as position * price

# ── early exposure carve (apply filter right after loading) ────────────────
# When set (typically one of EXPOSURE_FRACS, e.g. 0.5), per-wallet capture
# thresholds T_{w,f} are fit on the wallet's train BUY fills via
# wallet_capture_thresholds, and ALL fills (BUY + SELL, train + test) with
# deployed_capital (position * price) < T are dropped from df_full before
# the carved metrics/selection below are computed. Full-stream (uncarved)
# wallet stats are also computed BEFORE the carve, so both stat sets are
# available. Wallets with no positive train copyable_pnl get no threshold
# and are kept whole. None disables the filter (carved == full stats).
EXPOSURE_FILTER_FRAC = 0.2


DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)


## Derive train/test dates from data

## Full-stream wallet metrics for copy-trigger selection

Build per-wallet metrics on the full training trade stream (BUY + SELL).
Copyability is considered here (stage1), not in stage0 filtering.


In [6]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()

print(mdf.groupby('primary_tag').size().sort_values(ascending=False).head(20))

_tagset = set(TAGS)
mdf = mdf[
    (True if TAGS is None else mdf['tags'].apply(lambda tags: bool(_tagset & set(tags))))
    & ~(mdf['primary_tag'].isin(['Crypto']))
    & ((TAGS is None) or mdf["tags"].apply(lambda tags: any(tag in TAGS for tag in tags)))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)

print(f'selected markets for tags {TAGS}: {len(mdf)}')

primary_tag
Sports         3678528
Crypto         1082612
Weather         191014
Politics        103129
Finance          58994
Awards           12364
Movies           11521
AI                8498
Music             7778
Culture           6947
All               3808
Esports           3699
Business          3452
MrBeast           2302
Science           1868
Iran              1788
Celebrities       1447
Trump             1259
Tech              1018
Elections          928
dtype: int64
selected markets for tags {'Sports'}: 3265838


In [7]:
# Load full processed trades for the full-stream path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat(
    [pd.read_parquet(f).merge(mdf, on="condition_id", how="inner") for f in trade_files], ignore_index=True
    )

print(f"Full loaded df_full size: {len(df_full)}")

# df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

print(f"rows after mdf merge (BUY + SELL): {len(df_full):,}")

lead = (
    pd.to_datetime(df_full["last_condition_trade_ts"], utc=True, errors="coerce")
    - df_full["dt"]
)
df_full = df_full[lead <= pd.Timedelta(days=MAX_LEAD_DAYS)].copy()
print(
    f"Lead filter (<= {MAX_LEAD_DAYS}d before resolution): "
    f"{len(df_full):,} trades"
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"] = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"] = df_full["quantity"].astype(float)

# Full-stream PnL / notional across BUY + SELL
df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)
TRAIN_END = pd.Timestamp("2026-08-01", tz="UTC")
df_full['is_train'] = df_full['last_condition_trade_ts'] <= TRAIN_END

_n_wallets = df_full['wallet'].nunique()
_train = df_full[df_full['is_train']]
_test_wallets = set(df_full.loc[~df_full['is_train'], 'wallet'].unique())
_train_stats = _train.groupby('wallet').agg(
    train_pnl=('pnl', 'sum'),
    train_markets=('condition_id', 'nunique'),
)
del _train
_eligible = _train_stats.index[
    _train_stats.index.isin(_test_wallets)
    & (_train_stats['train_pnl'] >= 1000)
    & (_train_stats['train_markets'] >= 20)
]
df_full = df_full[df_full['wallet'].isin(_eligible)].copy()
print(f"Basic wallet filter: {df_full['wallet'].nunique():,} of {_n_wallets:,} wallets kept")

# FULL (uncarved) wallet metrics — computed before the exposure carve so
# both stat sets exist: full stats are passed to the selection filters,
# carved stats keep driving the execution/diagnostics stream below.
df_all_train_full = df_full[df_full["is_train"]].copy()
wallet_vol_all_full, _ = compute_wallet_metrics(df_all_train_full)
del df_all_train_full

df_buy_train_full = df_full[df_full["is_train"] & (df_full["side"] == "BUY")].copy()
wallet_vol_train_full, _ = compute_wallet_metrics(df_buy_train_full)
del df_buy_train_full

wallet_vol_train_full = wallet_vol_train_full.merge(
    wallet_vol_all_full[["wallet", "max_drawdown", "total_pnl"]].rename(
        columns={"max_drawdown": "max_drawdown_all", "total_pnl": "total_pnl_all"}
    ),
    on="wallet",
    how="left",
)
wallet_vol_train_full["max_drawdown_all_to_pnl_all"] = np.where(
    wallet_vol_train_full["total_pnl_all"] > 0,
    wallet_vol_train_full["max_drawdown_all"] / wallet_vol_train_full["total_pnl_all"],
    float("nan"),
)

if EXPOSURE_FILTER_FRAC is not None:
    from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

    df_full["deployed_capital"] = df_full["position"] * df_full["price"]

    _fit = df_full[df_full["is_train"] & (df_full["side"] == "BUY")]
    _capture = wallet_capture_thresholds(
        _fit, (EXPOSURE_FILTER_FRAC,), EXPOSURE_PNL_COL, "deployed_capital"
    )
    del _fit
    _T = df_full["wallet"].map(_capture.set_index("wallet")["threshold"]).fillna(0.0)

    _n = len(df_full)
    df_full = df_full[
        df_full["deployed_capital"].notna() & (df_full["deployed_capital"] >= _T)
    ].copy()
    print(
        f"Exposure carve (frac {EXPOSURE_FILTER_FRAC:g}): "
        f"{len(df_full):,} of {_n:,} trades kept"
    )

# TRAIN_START = TRAIN_END - pd.Timedelta(days=30)
# df_full = df_full[df_full['dt'] >= TRAIN_START]

# Carved all-trades (BUY + SELL) wallet metrics: drawdown incl. SELL legs
df_all_train = df_full[df_full["is_train"]].copy()
wallet_vol_all, _ = compute_wallet_metrics(df_all_train)
del df_all_train

# Primary stream stays BUY-only (carved execution/diagnostics stream)
df_full = df_full[df_full["side"] == "BUY"].copy().reset_index(drop=True)

df_slice = df_full[df_full["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
del df_slice

wallet_vol_train = wallet_vol_train.merge(
    wallet_vol_all[["wallet", "max_drawdown", "total_pnl"]].rename(
        columns={"max_drawdown": "max_drawdown_all", "total_pnl": "total_pnl_all"}
    ),
    on="wallet",
    how="left",
)
wallet_vol_train["max_drawdown_all_to_pnl_all"] = np.where(
    wallet_vol_train["total_pnl_all"] > 0,
    wallet_vol_train["max_drawdown_all"] / wallet_vol_train["total_pnl_all"],
    float("nan"),
)
print(len(wallet_vol_train))
print(len(wallet_vol_train_full))


Full loaded df_full size: 25212960
rows after mdf merge (BUY + SELL): 25,212,960
Lead filter (<= 2d before resolution): 24,384,903 trades
Basic wallet filter: 2,495 of 15,495 wallets kept


KeyError: 'copyable_qty_20m_100'

In [ ]:
len(df_full)

In [ ]:
daily_stats = (
    df_full.groupby(pd.Grouper(key="dt", freq="D"))
    .agg(
        trade_count=("price", "size"),
        total_volume=("price", lambda s: (s * df_full.loc[s.index, "quantity"]).sum()),
    )
    .reset_index()
)

# import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.graph_objects as go


# Create plot with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=daily_stats["dt"],
        y=daily_stats["trade_count"],
        name="Number of trades",
        mode="lines+markers",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=daily_stats["dt"],
        y=daily_stats["total_volume"],
        name="Total volume (price × qty)",
        mode="lines+markers",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Daily Trade Count and Total Volume",
    xaxis_title="Date",
    template="plotly_white",
    hovermode="x unified",
)

fig.update_yaxes(title_text="Number of trades", secondary_y=False)
fig.update_yaxes(title_text="Total volume", secondary_y=True)

fig.show()


In [ ]:
len(df_full[df_full["is_train"]])

In [ ]:
# mdf = load_markets_processed()
# df_full = pd.concat(
#     [
#         (
#             pd.read_parquet(f)
#             .merge(mdf, on="condition_id", how="inner")
#             .assign(dt=lambda df: pd.to_datetime(df["dt"], utc=True))
#             .loc[lambda df: df["dt"] >=  datetime.datetime(2026, 7, 1, tzinfo=datetime.timezone.utc)]
#         )
#         for f in trade_files
#     ],
#     ignore_index=True,
# )


In [ ]:
print (df_full.columns)
print(df_full.index.names)

In [ ]:
_copy_scale = np.minimum(2, df_full['avail_copy_qty_5m_100'] / df_full['quantity'])
df_full['copyable_qty_2x'] = df_full['copyable_qty_5m_100'] * _copy_scale
df_full['copyable_pnl_2x'] = df_full['copyable_pnl'] * _copy_scale

_copy_scale = np.minimum(0.5, df_full['avail_copy_qty_5m_100'] / df_full['quantity'])

df_full['copyable_qty_0.5x'] = df_full['copyable_qty_5m_100'] * _copy_scale
df_full['copyable_pnl_0.5x'] = df_full['copyable_pnl'] * _copy_scale

In [ ]:
# ── Copyable PnL variants ─────────────────────────────────────────────────────
# Per-fill copyable PnL under different lookback-window / fill-price assumptions.
# Computed in stage1 for now (stage0 may later export these per-fill columns).
COPYABLE_PNL_BASE = "copyable_pnl"
COPYABLE_VARIANTS = {
    # "5m_095": dict(qty_col="copyable_qty_5m_095", fill_factor=0.95),
    "20m_100": dict(qty_col="copyable_qty_20m_100", fill_factor=1.0),
    "2x": dict(qty_col="copyable_qty_2x", fill_factor=1.0),
    # "0.5x": dict(qty_col="copyable_qty_0.5x", fill_factor=1.0),
}

for key, cfg in COPYABLE_VARIANTS.items():
    qty = df_full[cfg["qty_col"]]
    df_full[f"copyable_pnl_{key}"] = np.where(
        df_full["side"] == "BUY",
        (df_full["final_price"] - df_full["price"] * cfg["fill_factor"]) * qty,
        (df_full["price"] / cfg["fill_factor"] - df_full["final_price"]) * qty,
    )

COPYABLE_PNL_COLS = [COPYABLE_PNL_BASE] + [f"copyable_pnl_{k}" for k in COPYABLE_VARIANTS]
COPYABLE_PNL_VARIANTS = COPYABLE_PNL_COLS[1:]


In [ ]:
df_full['copyable_notional'] = df_full['notional'] * (df_full['copyable_pnl'] / df_full['pnl'].replace(0, np.nan))
df_full['roi'] = df_full['pnl'] / df_full['notional'].replace(0, np.nan)
df_full['copyable_roi'] = df_full['copyable_pnl'] / df_full['copyable_notional'].replace(0, np.nan)

df_full.head()

In [ ]:
wallet_vol_test, _ = compute_wallet_metrics(df_full[~df_full["is_train"]].copy())

In [ ]:
df_full['bucket'] = df_full['dt'].dt.round("5min")

In [ ]:
# # WALLET COMBINATION CELL

# df = df_full[
#     (df_full['wallet'].isin(wallet_cohorts['predicting_group']['wallet']) | df_full['wallet'].isin(wallet_cohorts['copyable_group']['wallet']) | df_full['wallet'].isin(WALLET_FILTER))
#      & (df_full["is_train"])].copy()

# df['is_predicting_group'] = df['wallet'].isin(wallet_cohorts['predicting_group']['wallet'])
# df['is_filter_group'] = df['wallet'].isin(WALLET_FILTER)
# df['is_copyable_group'] = df['wallet'].isin(wallet_cohorts['copyable_group']['wallet']) & ~df['is_filter_group']



# print(f'predicting_size: {len(wallet_cohorts["predicting_group"]["wallet"])}')
# print(f'copyable_size: {len(wallet_cohorts["copyable_group"]["wallet"])}')

# print('\n')
# print('*** Total PnL / Copyable PnL before filtering ***')
# print(df[['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df["pnl"].sum() / df["notional"].sum()}')
# print(f'copyable_roi={df["copyable_pnl"].sum() / df["copyable_notional"].sum()}')


# print('\n')
# print('*** Predicting group PnL ***')
# print(df[df['is_predicting_group']][['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df[df["is_predicting_group"]]["pnl"].sum() / df[df["is_predicting_group"]]["notional"].sum()}')
# print(f'copyable_roi={df[df["is_predicting_group"]]["copyable_pnl"].sum() / df[df["is_predicting_group"]]["copyable_notional"].sum()}')

# print('\n')
# print('*** Copyable group PnL ***')
# print(df[df['is_copyable_group']][['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df[df["is_copyable_group"]]["pnl"].sum() / df[df["is_copyable_group"]]["notional"].sum()}')
# print(f'copyable_roi={df[df["is_copyable_group"]]["copyable_pnl"].sum() / df[df["is_copyable_group"]]["copyable_notional"].sum()}')

# df = df.groupby(['bucket', 'token_id', 'side']).agg(
#     copyable_pnl=('copyable_pnl', 'sum'),
#     wallet_count=('wallet', 'nunique'),
#     pnl=('pnl', 'sum'),
#     predicting_group_count=('is_predicting_group', 'sum'),
#     copyable_group_count=('is_copyable_group', 'sum'),
#     wallet_in_filter_count=('is_filter_group', 'sum'),
#     notional=('notional', 'sum'),
#     copyable_notional=('copyable_notional', 'sum'),
# )
# df.reset_index(inplace=True)
# df = df[
#     ((df['wallet_count'] > 1)
#     & (df['predicting_group_count'] <= 0)
#     & (df['copyable_group_count'] >= 0)
#      & (df['wallet_in_filter_count'] >= 0) )
#       | ( df['wallet_count'] == df['wallet_in_filter_count'])
#      ]

# print('\n')
# print('*** Combined PnL ***')
# print(df[['pnl', 'copyable_pnl']].sum())
# print(f'wallet_roi={df["pnl"].sum() / df["notional"].sum() if "notional" in df.columns else np.nan}')
# print(f'copyable_roi={df["copyable_pnl"].sum() / df["copyable_notional"].sum() if "copyable_notional" in df.columns else np.nan}')
# print(f'copyable_roi={df["copyable_pnl"].sum() / df["copyable_notional"].sum() if "copyable_notional" in df.columns else np.nan}')

In [ ]:
for _wv in (wallet_vol_train, wallet_vol_train_full):
    _wv['copyable_pnl_factor'] = np.clip(
        _wv['copyable_pnl'] / _wv['total_pnl'].replace(0, np.nan),
        0,
        1.0,
    ).fillna(0.0)
    _wv['copyable_roi'] = _wv['average_roi'] * _wv['copyable_pnl_factor']
    _wv['buy_copyable_roi'] = _wv['buy_copyable_pnl'] / _wv['buy_copyable_notional'].replace(0, np.nan)

# Keep sorting deterministic for downstream filtering/inspection.
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
wallet_vol_train_full.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train_full.reset_index(drop=True, inplace=True)


In [ ]:
wallet_cohorts = {}

In [ ]:
# Final wallet selection: threshold-based copyable + predicting groups
# COPYABLE_SCORE_MIN = -1
# PREDICTING_SCORE_MIN = -2

candidates = wallet_vol_train.copy()
candidates['buy_copyable_factor'] = candidates['buy_copyable_pnl'] / candidates['buy_pnl']

# Additional filter: keep only wallets active in the test period (> 0 test trades)
test_active_wallets = set(df_full.loc[~df_full['is_train'], 'wallet'].unique())
n_candidates = len(candidates)
candidates = candidates[candidates['wallet'].isin(test_active_wallets)].copy()
print(f"Test-active filter: kept {len(candidates):,} of {n_candidates:,} candidates")

for col, default in {
    'top10_pnl_pct': np.nan,
    'top_market_abs_pnl_pct': np.nan,
    'market_pnl_hhi': np.nan,
    'positive_bucket_share': np.nan,
}.items():
    if col not in candidates.columns:
        candidates[col] = default

for col in [
    'average_roi', 'median_roi', 'num_buckets', 'num_markets',
    'pnl_volatility', 'max_drawdown_to_pnl',
    'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
    'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl', 'total_notional',
    'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct', 'positive_bucket_share',
]:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "wallet_selection"))

from signal_lab.filters import (
    select_strategy_eligible,
    select_strategy_selection,
)

if "Esports" in TAGS:
    variant = "sports"
elif "Politics" in TAGS:
    variant = "politics"
else:
    variant = "weather"

# Two separate stat frames passed to the filter functions: the full
# (non-filtered) stats drive the mask rules for now; the carved stats are
# available as the second parameter for rule-level adjustments.
eligible_wallets = select_strategy_eligible(
    wallet_vol_train_full, wallet_vol_train, variant=variant
)
copyable_wallets = select_strategy_selection(
    wallet_vol_train_full, wallet_vol_train, variant=variant
)

eligible_base = candidates[candidates['wallet'].isin(eligible_wallets)].copy()
if eligible_base.empty:
    raise ValueError('No wallets passed base eligibility filters.')

copyable_candidates = eligible_base[eligible_base['wallet'].isin(copyable_wallets)].copy()
copyable_candidates['copyable_efficiency'] = copyable_candidates['copyable_pnl'].fillna(0.0) / (copyable_candidates['total_notional'].fillna(0.0) + 1.0)
copyable_candidates['copyable_dd_ratio'] = (
    copyable_candidates['max_copyable_drawdown_to_copyable_pnl']
    .fillna(copyable_candidates['max_drawdown_to_pnl'])
    .fillna(0.0)
)


wallet_cohorts['copyable_group'] = (
    copyable_candidates
    .reset_index(drop=True)
)

predicting_pool = eligible_base[
    ~eligible_base['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])
].copy()
wallet_cohorts['predicting_group'] = (
    predicting_pool
    .reset_index(drop=True)
)

overlap = set(wallet_cohorts['copyable_group']['wallet']) & set(wallet_cohorts['predicting_group']['wallet'])
if overlap:
    raise ValueError(f'Wallet groups must be disjoint; found overlap of {len(overlap)} wallets.')

wallet_cohorts['multi_filter'] = wallet_cohorts['copyable_group'].copy()

# Split the copyable group into 3 terciles by estimated_copyable_buy_sharpe.
# qcut raises when the values cannot form 3 unique bins (e.g. all-NaN sharpe
# after heavy filtering); skip the split and keep empty tercile cohorts so
# the summary print below still works.
_sharpe = wallet_cohorts['copyable_group']['estimated_copyable_buy_sharpe']
_SHARPE_LABELS = ['copyable_buy_sharpe_low', 'copyable_buy_sharpe_mid', 'copyable_buy_sharpe_high']
try:
    _terciles = pd.qcut(_sharpe, 3, labels=_SHARPE_LABELS)
except ValueError as _err:
    _terciles = None
    print(f"Sharpe terciles skipped: {_err}")
    print(
        f"  estimated_copyable_buy_sharpe: {_sharpe.notna().sum()}/{len(_sharpe)} valid, "
        f"{_sharpe.dropna().nunique()} distinct values"
    )
for _label in _SHARPE_LABELS:
    wallet_cohorts[_label] = (
        wallet_cohorts['copyable_group'][_terciles == _label].copy().reset_index(drop=True)
        if _terciles is not None
        else wallet_cohorts['copyable_group'].iloc[0:0].copy()
    )


print(f"Base-eligible wallets: {len(eligible_base):,}")
print(f"Copyable candidates: {len(copyable_candidates):,}")
print(f"Selected wallets (copyable group): {len(wallet_cohorts['copyable_group']):,} total copyable pnl {wallet_cohorts['copyable_group']['copyable_pnl'].sum()}")
print(f"Selected wallets (predicting group): {len(wallet_cohorts['predicting_group']):,} total predicting group copyable pnl {wallet_cohorts['predicting_group']['copyable_pnl'].sum()}")
print("Copyable sharpe terciles: " + ", ".join(f"{k}={len(wallet_cohorts[k])}" for k in ['copyable_buy_sharpe_low', 'copyable_buy_sharpe_mid', 'copyable_buy_sharpe_high']))
wallet_cohorts['copyable_group'].sort_values('estimated_copyable_buy_sharpe', key=abs, ascending=False).head(30)



In [ ]:
wallet_vol_train[wallet_vol_train['wallet'] == '0xc3973025163de81e367ad29c21e296e51684bb3b']

In [ ]:
print("TEST")
wallet_vol_test[wallet_vol_test['wallet'] == '0xc3973025163de81e367ad29c21e296e51684bb3b']

In [ ]:
selected_wallets_vol = wallet_vol_test[wallet_vol_test['wallet'].isin(wallet_cohorts['predicting_group'].wallet)]
print("TEST")
selected_wallets_vol.sort_values(by='copyable_pnl', key=abs, ascending=False).head(10)

In [ ]:
# ── Individual wallet inspection: PnL / copyable PnL variants / exposure ─────
# Per wallet: two stacked subplots (train / test) with cumulative wallet PnL,
# copyable-PnL variants (raw/unlimited) and their copy exposure (secondary
# axis). Additionally the bankroll-discipline carve variants per
# EXPOSURE_FRACS: T_{w,f} is fit on the wallet's TRAIN fills via
# wallet_capture_thresholds (smallest deployed capital capturing fraction `f`
# of the wallet's train copyable PnL; f == 1 is the raw baseline above), and
# the same thresholds carve the fills in both subplots.
import plotly.colors
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series
from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

INSPECT_WALLETS = [
    "0x3f176a240c7b6c0e470bf150f43fc1c883de7035",
    # append wallets here to inspect them individually
]

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}
LINE_COLORS = {
    "pnl": "#1f77b4",
    **{c: col for c, col in zip(COPYABLE_PNL_COLS, ["#ff7f0e", "#2ca02c", "#d62728", "#9467bd"])},
}

CARVE_FRACS = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)
FRAC_COLORS = {f: col for f, col in zip(CARVE_FRACS, plotly.colors.qualitative.Dark2)}

for wallet in INSPECT_WALLETS:
    w = df_full[df_full["wallet"] == wallet].copy()
    if w.empty:
        print(f"{wallet}: no fills in df_full, skipping")
        continue

    # per-fill copy exposure (price × copyable qty)
    for pnl_col, qty_col in PNL_TO_QTY.items():
        w[f"{pnl_col}_exposure"] = np.where(
            w[qty_col] > 0,
            w["price"] * w[qty_col],
            np.nan,
        )
    w["deployed_capital"] = w["position"] * w["price"]

    # carve thresholds T_{w,f}: fit on the wallet's train fills only
    fit = w[w["is_train"] & w["deployed_capital"].notna()][
        ["wallet", EXPOSURE_PNL_COL, "deployed_capital"]
    ]
    capture = wallet_capture_thresholds(fit, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital")
    carve_threshold = {}
    for frac in CARVE_FRACS:
        rows_ = capture[capture["frac"] == frac]
        if not rows_.empty:
            carve_threshold[frac] = float(rows_["threshold"].iloc[0])
    if len(carve_threshold) < len(CARVE_FRACS):
        print(f"{wallet}: no positive train {EXPOSURE_PNL_COL} — some/all carve variants skipped")

    fig = make_subplots(
        rows=2,
        cols=1,
        subplot_titles=("Train", "Test"),
        specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    )

    for row, (period, mask) in enumerate(
        [("train", w["is_train"]), ("test", ~w["is_train"])], start=1
    ):
        sub = w[mask].sort_values("dt")
        if sub.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=sub["dt"],
                y=sub["pnl"].cumsum(),
                mode="lines",
                name="pnl",
                legendgroup="pnl",
                showlegend=(row == 1),
                line=dict(color=LINE_COLORS["pnl"], width=2),
            ),
            row=row, col=1, secondary_y=False,
        )
        for c in COPYABLE_PNL_COLS:
            fig.add_trace(
                go.Scatter(
                    x=sub["dt"],
                    y=sub[c].cumsum(),
                    mode="lines",
                    name=c,
                    legendgroup=c,
                    showlegend=(row == 1),
                    line=dict(color=LINE_COLORS[c], width=2, dash="dash"),
                ),
                row=row, col=1, secondary_y=False,
            )
            exp = copyable_exposure_series(sub, f"{c}_exposure")
            if not exp.empty:
                fig.add_trace(
                    go.Scatter(
                        x=exp["dt"],
                        y=exp["exposure"],
                        mode="lines",
                        name=f"{c} exposure",
                        legendgroup=f"{c}_exp",
                        showlegend=(row == 1),
                        line=dict(color=LINE_COLORS[c], width=1.5, dash="dot"),
                        opacity=0.75,
                    ),
                    row=row, col=1, secondary_y=True,
                )

        # carved copyable PnL + exposure per bankroll frac (train-fit T_{w,f})
        for frac in CARVE_FRACS:
            if frac not in carve_threshold:
                continue
            carved = sub[
                sub["deployed_capital"].notna()
                & (sub["deployed_capital"] >= carve_threshold[frac])
            ]
            if carved.empty:
                continue
            color = FRAC_COLORS[frac]
            for c in COPYABLE_PNL_COLS:
                group = f"{c} (frac {frac:g})"
                fig.add_trace(
                    go.Scatter(
                        x=carved["dt"],
                        y=carved[c].cumsum(),
                        mode="lines",
                        name=group,
                        legendgroup=f"{c}_frac{frac:g}",
                        showlegend=(row == 1),
                        line=dict(color=color, width=1.5, dash="dash"),
                        opacity=0.9,
                    ),
                    row=row, col=1, secondary_y=False,
                )
                exp = copyable_exposure_series(carved, f"{c}_exposure")
                if not exp.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=exp["dt"],
                            y=exp["exposure"],
                            mode="lines",
                            name=f"{c} exposure (frac {frac:g})",
                            legendgroup=f"{c}_exp_frac{frac:g}",
                            showlegend=(row == 1),
                            line=dict(color=color, width=1.5, dash="dot"),
                            opacity=0.7,
                        ),
                        row=row, col=1, secondary_y=True,
                    )

    fig.update_layout(
        title=f"Wallet {wallet} — cumulative PnL, copyable PnL (variants + carve fracs) and exposure",
        template="plotly_white",
        height=800,
    )
    fig.update_yaxes(title_text="Cumulative PnL (USDC)", secondary_y=False)
    fig.update_yaxes(title_text="Exposure (USDC)", secondary_y=True)
    fig.show(renderer="browser")


In [ ]:
df_full['is_copyable'] = df_full['avail_copy_count_5m_100'] >=1

In [ ]:
df_full[df_full['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])] \
    .groupby('wallet') \
    .agg(
        trade_count=('wallet', 'count'),
        copyable_count=('is_copyable', 'sum')
    ) \
    .assign(copyable_ratio=lambda x: x['copyable_count'] / x['trade_count']) \
    .reset_index().sort_values('trade_count', ascending=False).head(5)

In [ ]:
if(WALLET_FILTER is not None):
    wallet_cohorts['copyable_group'] = wallet_cohorts['copyable_group'].query("wallet in @WALLET_FILTER").copy().reset_index(drop=True)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [ ]:
filter_wallets = set(wallet_cohorts['multi_filter']["wallet"])
df_test =  df_full[
    (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(2)

In [ ]:
wallets = set(wallet_cohorts['copyable_group']['wallet'])

df = df_full #[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

In [ ]:
wallet_set = set(wallet_cohorts['copyable_group']['wallet'])
print(f"Copyable wallets: {len(wallet_set)} with fills {df_wallets[df_wallets['wallet'].isin(wallet_set)].shape[0]} with total PnL: {df_wallets[df_wallets['wallet'].isin(wallet_set)]['pnl'].sum():,.2f}")
df_wallets[df_wallets['wallet'].isin(wallet_set)]['trade_pnl'].sum()

In [ ]:
df_wallets[df_wallets['wallet'].isin(wallet_set)].groupby('wallet').agg(
    pnl=('trade_pnl', 'sum'),
    notional=('notional', 'sum'),
    trades=('wallet', 'count'),
    copyable_pnl=('copyable_pnl', 'sum'),
    copyable_qty_5m_100=('copyable_qty_5m_100', 'sum'),
    ).sort_values(by="pnl", key=np.abs, ascending=False).head(20)

In [ ]:
# ── Copyable-group test stats: wallet vs copyable variants ──────────────────
# Cohort-level stats on the test period for the copyable_group: the wallets'
# own PnL (trade_pnl) and each copyable PnL variant. ROI uses the capital
# deployed (sum of per-fill exposure). Sharpe is the annualized daily-PnL
# Sharpe, zero-filled across the test date range. Exposure quantiles come
# from the capital-at-risk time profile (fills add exposure, released 1d
# after the contract's last trade).
#
# In addition to the full rows, each variant also gets a row per exposure
# carve (`EXPOSURE_FRACS`, frac < 1). The carve is a bankroll-discipline
# filter: per wallet keep the fills whose deployed capital
# (position * price, the post-trade USDC value of the book on that
# condition/outcome) is >= T_{w,f}, where T is fit on TRAIN via
# wallet_capture_thresholds (the smallest exposure whose retained fills
# capture fraction `f` of the wallet's train copyable PnL; f == 1 keeps
# everything). The same train-fit thresholds are applied to the test fills;
# wallets with no positive train PnL get no threshold and are kept whole.

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series
from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

df_full["deployed_capital"] = df_full["position"] * df_full["price"]

_copyable_wallets = wallet_cohorts["copyable_group"]["wallet"]
test_fills = df_full[
    (~df_full["is_train"]) & (df_full["wallet"].isin(_copyable_wallets))
].copy()
train_fills = df_full[
    (df_full["is_train"]) & (df_full["wallet"].isin(_copyable_wallets))
].copy()

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}

for key, qty_col in PNL_TO_QTY.items():
    test_fills[f"{key}_exposure"] = test_fills["price"] * test_fills[qty_col]


_capture = wallet_capture_thresholds(
    train_fills, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital"
)


def _annualized_daily_sharpe(daily: pd.Series, periods_per_year: float = 365.0) -> float:
    if len(daily.dropna()) < 2:
        return float("nan")
    full = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    s = daily.reindex(full, fill_value=0.0)
    sd = s.std()
    if sd == 0:
        return float("nan")
    return float(s.mean() / sd * np.sqrt(periods_per_year))


EXPOSURE_QUANTILES = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
CARVE_FRACS = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)

variant_specs = [("wallet", "trade_pnl", "notional")]
variant_specs += [(c, c, f"{c}_exposure") for c in COPYABLE_PNL_COLS]


def _stats_row(sub: pd.DataFrame, label: str, pnl_col: str, exp_col: str, n_total: int) -> dict:
    total_pnl = sub[pnl_col].sum()
    total_capital = sub[exp_col].sum()
    daily = sub.assign(_d=sub["dt"].dt.floor("D")).groupby("_d")[pnl_col].sum()
    profile = copyable_exposure_series(sub, exp_col, bucket_freq="D")
    exp_qs = {
        f"exp_p{int(q * 100)}": (
            profile["exposure"].quantile(q) if not profile.empty else float("nan")
        )
        for q in EXPOSURE_QUANTILES
    }
    return {
        "variant": label,
        "total_pnl": total_pnl,
        "total_roi": (
            total_pnl / total_capital
            if pd.notna(total_capital) and total_capital > 0
            else float("nan")
        ),
        "sharpe": _annualized_daily_sharpe(daily),
        "kept_rows": len(sub),
        "kept_rows_pct": len(sub) / n_total if n_total else float("nan"),
        **exp_qs,
        "exp_max": profile["exposure"].max() if not profile.empty else float("nan"),
    }


n_total = len(test_fills)
stat_rows = []
for label, pnl_col, exp_col in variant_specs:
    stat_rows.append(_stats_row(test_fills, label, pnl_col, exp_col, n_total))
    for frac in CARVE_FRACS:
        T = test_fills["wallet"].map(
            _capture[_capture["frac"] == frac].set_index("wallet")["threshold"]
        ).fillna(0.0)
        carved = test_fills[
            test_fills["deployed_capital"].notna() & (test_fills["deployed_capital"] >= T)
        ]
        if carved.empty:
            continue
        stat_rows.append(_stats_row(carved, f"{label} (frac {frac:g})", pnl_col, exp_col, n_total))

copyable_group_test_stats = pd.DataFrame(stat_rows)
copyable_group_test_stats


In [ ]:
MARKET = '0xaabf22a03217285585592f3082f27e2b0f4c1cee2bb4003e84817100bbcf1938'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty_5m_100=('copyable_qty_5m_100', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty_5m_100', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

In [ ]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [ ]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

In [ ]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [ ]:
import plotly.graph_objects as go

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series

print(f"copyable_group wallets: {len(wallet_cohorts['copyable_group']):,}")
print(f"predicting_group wallets: {len(wallet_cohorts['predicting_group']):,}")

pd.set_option('display.max_rows', 100)

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}

plot_rows = []
exposure_rows = []
for group_label, cohort_name in [('copyable', 'copyable_group'), ('predicting', 'predicting_group')]:
    wallets = set(wallet_cohorts[cohort_name]['wallet'])
    group_trades = df_full[
        (df_full['wallet'].isin(wallets))
        & (~df_full['is_train'])
    ][['dt', 'side', 'price', 'pnl', 'last_condition_trade_ts', *COPYABLE_PNL_COLS, *PNL_TO_QTY.values()]].copy()

    # per-fill copy exposure (price × copyable qty, BUY only)
    for pnl_col, qty_col in PNL_TO_QTY.items():
        group_trades[f'{pnl_col}_exposure'] = np.where(
            group_trades[qty_col] > 0,
            group_trades['price'] * group_trades[qty_col],
            np.nan,
        )

    daily = (
        group_trades
        .assign(plot_dt=group_trades['dt'].dt.floor('D'))
        .groupby('plot_dt', as_index=False)[['pnl', *COPYABLE_PNL_COLS]]
        .sum()
        .sort_values('plot_dt')
        .reset_index(drop=True)
    )
    daily['cum_trade_pnl'] = daily['pnl'].cumsum()
    for c in COPYABLE_PNL_COLS:
        daily[f'cum_{c}'] = daily[c].cumsum()
    daily['wallet_group'] = group_label
    plot_rows.append(daily)

    for c in COPYABLE_PNL_COLS:
        exp = copyable_exposure_series(group_trades, f'{c}_exposure', bucket_freq='D')
        exp['wallet_group'] = group_label
        exp['variant'] = c
        exposure_rows.append(exp)

group_perf = pd.concat(plot_rows, ignore_index=True)
group_exposure = pd.concat(exposure_rows, ignore_index=True)

fig = go.Figure()
colors = {'copyable': '#1f77b4', 'predicting': '#ff7f0e'}
dashes = ['dash', 'dot', 'dashdot']
for group_label in ['copyable', 'predicting']:
    g = group_perf[group_perf['wallet_group'] == group_label]
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_trade_pnl'],
            mode='lines',
            name=f'{group_label} group trade pnl',
            line=dict(color=colors[group_label], width=2),
        )
    )
    for i, c in enumerate(COPYABLE_PNL_COLS):
        fig.add_trace(
            go.Scatter(
                x=g['plot_dt'],
                y=g[f'cum_{c}'],
                mode='lines',
                name=f'{group_label} {c} (raw, unlimited)',
                line=dict(color=colors[group_label], width=2, dash=dashes[i % len(dashes)]),
            )
        )
    for c in COPYABLE_PNL_COLS:
        e = group_exposure[
            (group_exposure['wallet_group'] == group_label)
            & (group_exposure['variant'] == c)
        ]
        fig.add_trace(
            go.Scatter(
                x=e['dt'],
                y=e['exposure'],
                mode='lines',
                name=f'{group_label} {c} exposure',
                line=dict(color=colors[group_label], width=1.5, dash='dot'),
                opacity=0.75,
                yaxis='y2',
            )
        )

fig.update_layout(
    title='Stage1 wallet-group performance on test period — copyable PnL raw/unlimited, exposure on right axis',
    xaxis_title='Date',
    yaxis_title='Cumulative PnL (USDC)',
    yaxis2=dict(title='Exposure (USDC)', overlaying='y', side='right', showgrid=False),
    template='plotly_white',
)
fig.show()

In [ ]:
copyable_wallets = wallet_cohorts['copyable_group'].copy().assign(wallet_group='copyable')
predicting_wallets = wallet_cohorts['predicting_group'].copy().assign(wallet_group='predicting')

selected_wallets = (
    pd.concat([copyable_wallets, predicting_wallets], ignore_index=True)
    .drop_duplicates(subset=['wallet', 'wallet_group'])
    .reset_index(drop=True)
)

selected_wallets.to_parquet(WORKSPACE_DIR / 'selected_wallets_v2.parquet', index=False)
print(f"Saved selected wallets: {len(selected_wallets):,}")
print(selected_wallets['wallet_group'].value_counts().to_dict())


In [ ]:
len(df_wallets[df_wallets['is_train'] == False])

## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [ ]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [ ]:
df_fills = df_full.copy()
df_fills["end_date_iso"] = pd.to_datetime(df_fills["last_condition_trade_ts"], utc=True).dt.floor("D")

# per-variant fill exposure: capital needed to fill the copyable qty at the fill price
PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}
for pnl_col, qty_col in PNL_TO_QTY.items():
    df_fills[f"{pnl_col}_exposure"] = np.where(
        df_fills[qty_col] > 0,
        df_fills["price"] * df_fills[qty_col],
        np.nan,
    )

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)


In [ ]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    # & (TAGS is None or df_fills['primary_tag'].isin(TAGS))
    & (df_fills["is_train"] == False)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
# filter_fills = filter_fills[filter_fills['copyable_qty_5m_100'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

In [ ]:
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 1000)

In [ ]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor("5min")

MAX_EXPOSURE = 100000

df = filter_fills.copy()

# scale each copyable variant to the per-fill MAX_EXPOSURE capital budget
for c in COPYABLE_PNL_COLS:
    exp_col = f"{c}_exposure"
    scale = np.minimum(1, MAX_EXPOSURE / df[exp_col].replace(0, np.nan))
    df[f"scaled_{c}"] = df[c] * scale
    df[f"scaled_{exp_col}"] = df[exp_col] * scale

df.head(1)


In [ ]:
# ── exposure + PnL over time, per copyable variant ────────────────────────────
agg_dict = {"wallet_pnl": ("trade_pnl", "sum")}
agg_dict.update({f"scaled_{c}": (f"scaled_{c}", "sum") for c in COPYABLE_PNL_COLS})
agg_dict.update({f"scaled_{c}_exposure": (f"scaled_{c}_exposure", "sum") for c in COPYABLE_PNL_COLS})
df_plot = df.groupby("end_date_iso").agg(**agg_dict).reset_index()

df.sort_values("bucket", inplace=True)
df["wallet_pnl_cumsum"] = df["trade_pnl"].cumsum()
for c in COPYABLE_PNL_COLS:
    df[f"scaled_{c}_cumsum"] = df[f"scaled_{c}"].cumsum()


def exposure_timeline(sub, c):
    """Cumulative exposure over time for a subset frame / copyable variant.

    Fills add exposure at buy time; positions resolve at EOD of end_date_iso
    (+1 day), releasing the exposure (negative delta)."""
    exp_col = f"scaled_{c}_exposure"
    daily = sub.groupby("end_date_iso")[exp_col].sum().reset_index()
    resolution = daily.rename(columns={exp_col: "exposure_delta"})
    resolution["dt"] = pd.to_datetime(resolution["end_date_iso"]) + pd.Timedelta(days=1)
    resolution["exposure_delta"] = -resolution["exposure_delta"]
    resolution["variant"] = c
    buy = sub[["bucket", exp_col]].rename(columns={"bucket": "dt", exp_col: "exposure_delta"})
    buy["variant"] = c
    ev = pd.concat(
        [resolution[["dt", "variant", "exposure_delta"]], buy[["dt", "variant", "exposure_delta"]]],
        ignore_index=True,
    )
    ev.sort_values("dt", inplace=True)
    ev["exposure"] = ev.groupby("variant")["exposure_delta"].cumsum()
    return ev


variant_exposure = pd.concat(
    [exposure_timeline(df, c) for c in COPYABLE_PNL_COLS], ignore_index=True)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["bucket"],
    y=df["wallet_pnl_cumsum"],
    mode="lines",
    name="wallet_pnl",
))

for c in COPYABLE_PNL_COLS:
    fig.add_trace(go.Scatter(
        x=df["bucket"],
        y=df[f"scaled_{c}_cumsum"],
        mode="lines",
        name=f"scaled_{c}",
        line=dict(dash="dash"),
    ))

for c in COPYABLE_PNL_COLS:
    exp_df = variant_exposure[variant_exposure["variant"] == c]
    fig.add_trace(go.Scatter(
        x=exp_df["dt"],
        y=exp_df["exposure"],
        mode="lines",
        name=f"exposure ({c})",
        line=dict(dash="dot"),
    ))

# ── deployed-capital carved copies ───────────────────────────────────────────
# Bankroll-discipline carve: per wallet keep the fills whose deployed capital
# (position * price, post-trade USDC book value on that condition / outcome)
# is >= T_{w,f}, with T fit on TRAIN via wallet_capture_thresholds (the
# smallest exposure capturing fraction `f` of the wallet's train copyable PnL;
# f == 1 keeps everything). The same train-fit T is applied to the test frame
# `df`; PnL / exposure lines are rebuilt on the carved subset.
# frac == 1 is the uncarved copy_all baseline (already drawn above).
import plotly.colors as _pl_colors

df_fills["deployed_capital"] = df_fills["position"] * df_fills["price"]
df["deployed_capital"] = df["position"] * df["price"]

_fit = df_fills[
    df_fills["wallet"].isin(wallet_cohorts["multi_filter"]["wallet"])
    & (df_fills["side"] == "BUY")
    & ((TAGS is None) or df_fills["primary_tag"].isin(TAGS))
    & df_fills["is_train"]
    & df_fills["deployed_capital"].notna()
][["wallet", EXPOSURE_PNL_COL, "deployed_capital"]].copy()

_capture = wallet_capture_thresholds(
    _fit, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital"
)

_exposed_fracs = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)
FRAC_COLOR = {f: _pl_colors.qualitative.Plotly[i % 10] for i, f in enumerate(_exposed_fracs)}

carve_sizes = {}   # frac -> kept rows (shared across all copyable variants)
carve_index = {}   # frac -> kept index

for frac in _exposed_fracs:
    T = df["wallet"].map(
        _capture[_capture["frac"] == frac].set_index("wallet")["threshold"]
    ).fillna(0.0)
    carved = df[
        df["deployed_capital"].notna() & (df["deployed_capital"] >= T)
    ].sort_values("bucket")
    if carved.empty:
        continue
    color = FRAC_COLOR[frac]
    for c in COPYABLE_PNL_COLS:
        fig.add_trace(go.Scatter(
            x=carved["bucket"],
            y=carved[f"scaled_{c}"].cumsum(),
            mode="lines",
            name=f"scaled_{c} (frac {frac:g})",
            line=dict(color=color, dash="dash", width=2.0),
            opacity=0.95,
        ))
        expos = exposure_timeline(carved, c)
        fig.add_trace(go.Scatter(
            x=expos["dt"],
            y=expos["exposure"],
            mode="lines",
            name=f"exposure ({c}) frac {frac:g}",
            line=dict(color=color, dash="dot", width=1.5),
            opacity=0.75,
        ))
    carve_sizes[frac] = len(carved)
    carve_index[frac] = carved.index

fig.show(renderer="browser")

# ── validation: how much does each carve actually keep? ─────────────────────
# With train-fit deployed-capital thresholds the carve is shared across all
# copyable variants (same fill set per frac). Show the retained fill share,
# deployed-capital share, and base copyable PnL share on the test frame.
print(f"total test fills (df): {len(df):,}")
validation_rows = []
for frac in _exposed_fracs:
    idx = carve_index.get(frac)
    if idx is None:
        continue
    carved = df.loc[idx]
    validation_rows.append({
        "frac": frac,
        "kept_rows": len(carved),
        "kept_rows_pct": len(carved) / len(df),
        "kept_deployed_cap_pct": (
            carved["deployed_capital"].sum() / df["deployed_capital"].sum()
            if df["deployed_capital"].sum() else float("nan")
        ),
        "kept_copyable_pnl_pct": (
            carved[COPYABLE_PNL_BASE].sum() / df[COPYABLE_PNL_BASE].sum()
            if df[COPYABLE_PNL_BASE].sum() else float("nan")
        ),
    })
pd.DataFrame(validation_rows).round(3)


In [ ]:
df_fills[(
    (df_fills['is_train'] == False)
    & (df_fills['wallet'].isin(wallet_set))
    & (df_fills['side'] == 'BUY')
          )].groupby(['primary_tag', 'side']).agg(
    copyable_pnl=('copyable_pnl', 'sum'),
    total_copyable_qty=('copyable_qty_5m_100', 'sum'),
    trade_pnl=('trade_pnl', 'sum'),
    trades=('condition_id', 'count'),
    condition_count=('condition_id', 'nunique'),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values('copyable_pnl', key=abs, ascending=False).reset_index().head(10)

In [ ]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Plot the copyable sharpe terciles in place of the combined copyable_group
    plot_cohorts = {k: v for k, v in wallet_cohorts.items() if k != 'copyable_group'}

    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        plot_cohorts,
        split_date=TRAIN_END,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period, unlimited copyable PnL + exposure)",
        pnl_cols=COPYABLE_PNL_COLS,
        exposure_fracs=EXPOSURE_FRACS,
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        plot_cohorts,
        split_date=TRAIN_END,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period, unlimited copyable PnL + exposure)",
        pnl_cols=COPYABLE_PNL_COLS,
        exposure_fracs=EXPOSURE_FRACS,
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")
